In [21]:
from pathlib import Path
import pandas as pd
from ete3 import NCBITaxa
from Bio import Entrez, SeqIO
from collections import defaultdict
from functools import lru_cache
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import requests

In [22]:
DATA_PATH = 'data/m_rna_entities_to_be_processed.pkl'
mrna_entities = pd.read_pickle(DATA_PATH)

In [23]:
print("Unique mRNA entities:", len(mrna_entities))
print("NaN in Raw_ID_key:", mrna_entities['Raw_ID_key'].isna().sum())
print("NaN in Symbol:", mrna_entities['Symbol'].isna().sum())
print("Species involved:", mrna_entities['Species'].nunique())

Unique mRNA entities: 213723
NaN in Raw_ID_key: 0
NaN in Symbol: 1
Species involved: 111


In [29]:
db_counts = defaultdict(int)
for buckets in mrna_entities['ID_buckets']:
    for db, ids in buckets.items():
        if db != 'RNAcentral':  # exclude placeholder
            db_counts[db] += len(set(ids))  # count unique IDs per entity

# Print summary
for db, count in db_counts.items():
    print(f"{db}: {count} unique IDs across entities")


miRBase: 0 unique IDs across entities
NCBI: 202217 unique IDs across entities
Ensembl: 1177 unique IDs across entities
circBase: 0 unique IDs across entities
tRFdb: 0 unique IDs across entities
GenBank: 0 unique IDs across entities
Other: 0 unique IDs across entities


# Transcripts map

### 1- NCBI IDs

In [46]:
# --- 1) Load gene2refseq fast (vectorized) ---
g2r = pd.read_csv(
    "data/gene2refseq",
    sep="\t", comment="#", header=None,
    usecols=[0, 1, 2, 3, 15],   # tax_id, GeneID, status, RNA_accession, Symbol
    names=["taxid", "GeneID", "status", "rna_acc", "Symbol"], dtype=str, low_memory=False
)
# keep mRNA RefSeq
g2r = g2r[(g2r["status"] != "SUPPRESSED") & (g2r["rna_acc"] != "-") & (g2r["rna_acc"].str.startswith(("NM_", "XM_"), na=False))].copy()

# --- Build mapping (taxid, GeneID) -> list of RefSeq mRNAs ---
refseq_map = (
    g2r.groupby(["taxid", "GeneID"])["rna_acc"]
    .agg(lambda s: sorted(set(s)))
    .reset_index()
)
refseq_map["taxid"] = refseq_map["taxid"].astype(str)
refseq_map["GeneID"] = refseq_map["GeneID"].astype(str)

# --- Create a dict for faster lookup ---
refseq_dict = {
    (row.taxid, row.GeneID): row.rna_acc
    for row in refseq_map.itertuples(index=False)
}

# --- Map RefSeq transcripts for rows with NCBI IDs ---
def get_refseq_transcripts(bucket, species):
    if not isinstance(bucket, dict):
        return None
    transcripts = []
    for geneid in bucket.get("NCBI", []):
        key = (str(species), str(geneid))
        transcripts.extend(refseq_dict.get(key, []))
    return sorted(set(transcripts)) if transcripts else None

# --- Apply to mrna_entities ---
mrna_entities["refseq"] = mrna_entities.apply(
    lambda row: get_refseq_transcripts(row["ID_buckets"], row["Species"]),
    axis=1
)

# --- Stats ---
n_refseq = mrna_entities["refseq"].notna().sum()
print(
    "Number of rows with RefSeq transcripts:", n_refseq, "/", len(mrna_entities),
    "Percentage:", 100 * n_refseq / len(mrna_entities)
)

Number of rows with RefSeq transcripts: 194559 / 213723 Percentage: 91.03325332322679


In [ ]:
mrna_entities

### 2-Ensemble IDs

In [53]:
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor

ENSEMBL_REST = "https://rest.ensembl.org"
HEADERS_JSON = {"Content-Type": "application/json"}
HEADERS_TXT = {"Content-Type": "text/plain"}

REQUEST_DELAY = 0.07   # ~14 req/sec (Ensembl safe limit)


# --- Extract Ensembl IDs ---
def extract_ensembl_ids(bucket):
    if not isinstance(bucket, dict):
        return []
    return [x for x in bucket.get("Ensembl", []) if isinstance(x, str) and x.startswith("ENS")]


mrna_entities["ensembl_ids"] = mrna_entities["ID_buckets"].apply(extract_ensembl_ids)

all_ids = sorted(set(
    tid for sub in mrna_entities["ensembl_ids"].dropna() for tid in sub
))

print("Total unique Ensembl IDs:", len(all_ids))


# --- Lookup each ID individually ---
def lookup_id(eid):

    url = f"{ENSEMBL_REST}/lookup/id/{eid}"

    for _ in range(3):
        r = requests.get(url, headers=HEADERS_JSON)

        if r.status_code == 200:
            return r.json()

        if r.status_code == 429:
            time.sleep(0.5)
            continue

        return None

    return None


metadata = {}

for i, eid in enumerate(all_ids, 1):

    if i % 100 == 0:
        print("Processed IDs:", i)

    data = lookup_id(eid)

    if data is not None:
        metadata[eid] = data

    time.sleep(REQUEST_DELAY)


print("Resolved IDs:", len(metadata))


# --- Build metadata dataframe ---
meta_df = pd.DataFrame.from_dict(metadata, orient="index").reset_index()
meta_df = meta_df.rename(columns={"index": "ensembl_id"})

print("Object type counts:")
print(meta_df["object_type"].value_counts())


# --- Separate genes and transcripts ---
gene_ids = meta_df.loc[meta_df["object_type"] == "Gene", "ensembl_id"].tolist()
tx_ids = meta_df.loc[meta_df["object_type"] == "Transcript", "ensembl_id"].tolist()

print("Gene IDs:", len(gene_ids))
print("Transcript IDs:", len(tx_ids))


# --- Resolve gene → transcripts ---
def lookup_gene_transcripts(gid):

    url = f"{ENSEMBL_REST}/lookup/id/{gid}?expand=1"

    for _ in range(3):

        r = requests.get(url, headers=HEADERS_JSON)

        if r.status_code == 200:

            data = r.json()

            tx = [
                t["id"]
                for t in data.get("Transcript", [])
                if t.get("biotype") == "protein_coding"
            ]

            return tx

        if r.status_code == 429:
            time.sleep(0.5)
            continue

        return []

    return []


gene_tx_map = {}

for i, gid in enumerate(gene_ids, 1):

    if i % 100 == 0:
        print("Resolved genes:", i)

    tx = lookup_gene_transcripts(gid)

    if tx:
        gene_tx_map[gid] = tx

    time.sleep(REQUEST_DELAY)


print("Genes with protein-coding transcripts:", len(gene_tx_map))


# --- Collect all transcript IDs ---
all_tx = set(tx_ids)

for v in gene_tx_map.values():
    all_tx.update(v)

all_tx = sorted(all_tx)

print("Total transcripts to fetch:", len(all_tx))


# --- Fetch cDNA sequences ---
def fetch_sequence(tid):

    url = f"{ENSEMBL_REST}/sequence/id/{tid}?type=cdna"

    for _ in range(3):

        r = requests.get(url, headers=HEADERS_TXT)

        if r.status_code == 200:
            return tid, r.text.strip()

        if r.status_code == 429:
            time.sleep(0.5)
            continue

        return tid, None

    return tid, None


with ThreadPoolExecutor(max_workers=4) as ex:
    seq_pairs = list(ex.map(fetch_sequence, all_tx))


ensembl_map = {k: v for k, v in seq_pairs if v}

print("Sequences retrieved:", len(ensembl_map))


# --- Map sequences back to mrna_entities ---
def get_ensembl_sequences(ids):

    if not ids:
        return None

    seqs = []

    for i in ids:

        if i in ensembl_map:
            seqs.append(ensembl_map[i])

        elif i in gene_tx_map:

            for tx in gene_tx_map[i]:

                if tx in ensembl_map:
                    seqs.append(ensembl_map[tx])

    return seqs if seqs else None


mrna_entities["ensembl_seq"] = mrna_entities["ensembl_ids"].apply(get_ensembl_sequences)


print(
    "Rows with Ensembl mRNA transcripts:",
    mrna_entities["ensembl_seq"].notna().sum(),
    "/",
    len(mrna_entities)
)

Total unique Ensembl IDs: 1177
Processed IDs: 100
Processed IDs: 200
Processed IDs: 300
Processed IDs: 400
Processed IDs: 500
Processed IDs: 600
Processed IDs: 700
Processed IDs: 800
Processed IDs: 900
Processed IDs: 1000
Processed IDs: 1100
Resolved IDs: 447
Object type counts:
object_type
Gene          409
Transcript     38
Name: count, dtype: int64
Gene IDs: 409
Transcript IDs: 38
Resolved genes: 100
Resolved genes: 200
Resolved genes: 300
Resolved genes: 400
Genes with protein-coding transcripts: 280
Total transcripts to fetch: 1374
Sequences retrieved: 1374
Rows with Ensembl mRNA transcripts: 318 / 213723


### 3- Symbol IDs - with RefSeq

In [55]:
# --- Select rows without any transcripts ---
missing_mask = (
    mrna_entities["refseq"].isna() &
    mrna_entities["ensembl_seq"].apply(lambda x: not x)
)

missing_rows = mrna_entities[missing_mask].copy()
print("Rows without any mapped transcripts:", len(missing_rows))

# --- Build Symbol -> RefSeq mapping ---
# g2r already contains (taxid, GeneID, rna_acc, Symbol)
# We'll build a dict: (taxid, Symbol) -> list of RefSeq mRNAs
symbol_map = (
    g2r.groupby(["taxid", "Symbol"])["rna_acc"]
    .agg(lambda s: sorted(set(s)))
    .reset_index()
)

symbol_map["taxid"] = symbol_map["taxid"].astype(str)
symbol_map["Symbol"] = symbol_map["Symbol"].astype(str)

symbol_dict = {
    (row.taxid, row.Symbol): row.rna_acc
    for row in symbol_map.itertuples(index=False)
}

# --- Function to map RefSeq via Symbol ---
def get_refseq_by_symbol(row):
    if not row["Symbol"] or not isinstance(row["Symbol"], str):
        return None
    key = (str(row["Species"]), row["Symbol"])
    return symbol_dict.get(key, None)

# --- Apply to missing rows ---
mrna_entities.loc[missing_mask, "refseq"] = missing_rows.apply(
    get_refseq_by_symbol,
    axis=1
)

# --- Stats ---
n_refseq_total = mrna_entities["refseq"].notna().sum()
print(
    "Number of rows with RefSeq transcripts after symbol mapping:",
    n_refseq_total, "/", len(mrna_entities),
    "Percentage:", 100 * n_refseq_total / len(mrna_entities)
)


Rows without any mapped transcripts: 18846
Number of rows with RefSeq transcripts after symbol mapping: 197251 / 213723 Percentage: 92.29282763202838


### Sequences fetch (RefSeq)

In [ ]:
import requests
import pandas as pd
import time
import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Parameters ---
BATCH_SIZE = 200
EFETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
CACHE_FILE = "data/refseq_sequences_cache.json"
MAX_WORKERS = 3  # ~3 requests/sec to stay under NCBI limits
RETRY = 3
DELAY_BETWEEN_REQS = 0.34  # ~3 requests/sec

# --- Load cache if exists ---
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        sequence_cache = json.load(f)
else:
    sequence_cache = {}

# --- Helper: fetch sequences for a batch of RefSeq IDs ---
def fetch_refseq_batch(batch):
    params = {
        "db": "nuccore",
        "id": ",".join(batch),
        "rettype": "fasta",
        "retmode": "text"
    }

    for _ in range(RETRY):
        r = requests.get(EFETCH_URL, params=params)
        if r.status_code == 200:
            break
        time.sleep(1)
    else:
        print(f"Failed batch: {batch[:5]}...")
        return {}

    # Parse fasta
    sequences = {}
    lines = r.text.splitlines()
    current_id = None
    seq_lines = []
    for line in lines:
        if line.startswith(">"):
            if current_id:
                sequences[current_id] = "".join(seq_lines)
            current_id = line.split()[0][1:]  # remove ">"
            seq_lines = []
        else:
            seq_lines.append(line.strip())
    if current_id:
        sequences[current_id] = "".join(seq_lines)
    time.sleep(DELAY_BETWEEN_REQS)
    return sequences

# --- Prepare all unique RefSeq IDs from mrna_entities ---
all_refseq_ids = sorted(set(
    rid for sub in mrna_entities["refseq"].dropna() for rid in sub
))

print(f"Total unique RefSeq IDs: {len(all_refseq_ids)}")

# --- Filter out already cached IDs ---
to_fetch = [rid for rid in all_refseq_ids if rid not in sequence_cache]
print(f"IDs to fetch: {len(to_fetch)}")

# --- Split into batches ---
batches = [to_fetch[i:i+BATCH_SIZE] for i in range(0, len(to_fetch), BATCH_SIZE)]

# --- Parallel fetch ---
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    future_to_batch = {ex.submit(fetch_refseq_batch, batch): batch for batch in batches}
    for future in as_completed(future_to_batch):
        batch_seq = future.result()
        sequence_cache.update(batch_seq)

        # --- Save cache incrementally ---
        with open(CACHE_FILE, "w") as f:
            json.dump(sequence_cache, f)

print(f"Total sequences cached: {len(sequence_cache)}")

# --- Map sequences back to rows ---
def get_refseq_sequences(row):
    ids = row["refseq"]
    if not ids:
        return None
    seqs = [sequence_cache.get(rid) for rid in ids if rid in sequence_cache]
    return seqs if seqs else None

mrna_entities["refseq_seq"] = mrna_entities["refseq"].apply(get_refseq_sequences)

# --- Stats ---
n_seq = mrna_entities["refseq_seq"].notna().sum()
print(f"Rows with RefSeq sequences: {n_seq} / {len(mrna_entities)}")

Total unique RefSeq IDs: 945364
IDs to fetch: 945364


In [ ]:
mrna_entities

In [ ]:
# --- Initialize sequences column with RefSeq sequences ---
mrna_entities["sequences"] = mrna_entities["refseq_seq"].copy()

# --- Add Ensembl sequences if present ---
def merge_ensembl(row):
    seqs = row["sequences"] if row["sequences"] else []
    if row.get("ensembl_seq"):
        seqs.extend(row["ensembl_seq"])
    return seqs if seqs else None

mrna_entities["sequences"] = mrna_entities.apply(merge_ensembl, axis=1)

# --- Count rows before filtering ---
print("Total rows before removing empty sequences:", len(mrna_entities))

# --- Remove rows with empty sequences column ---
mrna_entities = mrna_entities[mrna_entities["sequences"].apply(lambda x: x is not None and len(x) > 0)].copy()

# --- Count rows after filtering ---
print("Total rows after removing empty sequences:", len(mrna_entities))

### Summary of OLD results:
### Source of the IDs | RefSeq Retrieved (for now and just using gene2refseq)
- NCBI       202.217  |       195.540    #(searching with IDs and symbols) 
- NA          10.329  |        1.649     #(searching just with symbol)
- Ensembl      1.177  |         213      #(searching just with symbol)  
- Total      213.723  |       197.402    #(92.36% of all the sequences has retrieved at least one refseq)

### For NCBI IDs: Number of refseq: 195.540 / 202.217 Percentage: ~96.7%

# GeneID → GO terms

In [ ]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

print("GO fetch start:", datetime.now())

# --- Caches ---
go_cache = {}   # (ID, taxid) -> GO dict
raw_cache = {}  # full API response

# --- Normalize GO fields ---
def normalize_go(block):
    if not block:
        return []
    if isinstance(block, dict):
        block = [block]
    return [
        {
            "id": x.get("id"),
            "term": x.get("term"),
            "evidence": x.get("evidence"),
            "qualifier": x.get("qualifier"),
            "pubmed": x.get("pubmed"),
        }
        for x in block
        if x.get("id") and x.get("term")
    ]

# --- Fetch GO batch ---
def fetch_go_batch(pairs, source="NCBI"):
    if not pairs:
        return

    ids = [str(g) for g, _ in pairs]
    scope = "ensembl.gene" if source == "Ensembl" else None

    try:
        data = {"ids": ",".join(ids),
                "fields": "go,taxid,symbol,name,type_of_gene,genomic_pos,summary"}
        if scope:
            data["scope"] = scope

        r = requests.post(
            "https://mygene.info/v3/gene",
            data=data,
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=60,
        )
        r.raise_for_status()
    except Exception as e:
        print(f"{source} batch failed {ids[:5]}..: {e}")
        return

    resp = r.json()
    tax_map = {str(g): str(t) for g, t in pairs}

    for rec in resp:
        gid = str(rec.get("query"))
        taxid_api = str(rec.get("taxid")) if rec.get("taxid") else None
        if gid not in tax_map or taxid_api != tax_map[gid]:
            continue
        go = rec.get("go") or {}
        go_cache[(gid, tax_map[gid])] = {
            "BP": normalize_go(go.get("BP")),
            "MF": normalize_go(go.get("MF")),
            "CC": normalize_go(go.get("CC")),
        }
        raw_cache[(gid, tax_map[gid])] = rec

# --- Build ID + taxid pairs from mrna_entities ---
def get_id_pairs(row):
    pairs = []
    taxid = str(int(row["Species"]))
    bucket = row.get("ID_buckets", {})
    if not isinstance(bucket, dict):
        return pairs

    for source in ["NCBI", "Ensembl"]:
        ids = bucket.get(source, [])
        if isinstance(ids, (int, str)):
            ids = [ids]
        for gid in ids:
            if pd.notna(gid):
                pairs.append((str(gid), taxid))
    return pairs

all_pairs = []
for _, row in mrna_entities.iterrows():
    all_pairs.extend(get_id_pairs(row))
all_pairs = list({(g, t) for g, t in all_pairs})  # deduplicate

# --- Split by source for API ---
ncbi_pairs = [p for p in all_pairs if p[0].isdigit()]
ensembl_pairs = [p for p in all_pairs if not p[0].isdigit()]

# --- Batch helper ---
BATCH_SIZE = 1000
def batch_list(lst):
    return [lst[i:i+BATCH_SIZE] for i in range(0, len(lst), BATCH_SIZE)]

ncbi_batches = batch_list(ncbi_pairs)
ensembl_batches = batch_list(ensembl_pairs)

# --- Parallel fetch ---
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(fetch_go_batch, batch, "NCBI") for batch in ncbi_batches]
    futures += [executor.submit(fetch_go_batch, batch, "Ensembl") for batch in ensembl_batches]
    for f in as_completed(futures):
        f.result()

# --- Map GO back to DataFrame ---
def map_go(row):
    taxid = str(int(row["Species"]))
    bucket = row.get("ID_buckets", {})
    result = {"BP": [], "MF": [], "CC": []}
    for source in ["NCBI", "Ensembl"]:
        ids = bucket.get(source, [])
        if isinstance(ids, (int, str)):
            ids = [ids]
        for gid in ids:
            key = (str(gid), taxid)
            if key in go_cache:
                for k in ["BP", "MF", "CC"]:
                    result[k].extend(go_cache[key][k])
    # deduplicate by GO id
    for k in result:
        seen = set()
        result[k] = [x for x in result[k] if x["id"] not in seen and not seen.add(x["id"])]
    return result

mrna_entities["GO_raw"] = mrna_entities.apply(map_go, axis=1)
mrna_entities["GO_BP_count"] = mrna_entities["GO_raw"].apply(lambda d: len(d["BP"]))
mrna_entities["GO_MF_count"] = mrna_entities["GO_raw"].apply(lambda d: len(d["MF"]))
mrna_entities["GO_CC_count"] = mrna_entities["GO_raw"].apply(lambda d: len(d["CC"]))

# --- Keep raw payload if needed ---
def map_raw(row):
    taxid = str(int(row["Species"]))
    bucket = row.get("ID_buckets", {})
    for source in ["NCBI", "Ensembl"]:
        ids = bucket.get(source, [])
        if isinstance(ids, (int, str)):
            ids = [ids]
        for gid in ids:
            key = (str(gid), taxid)
            if key in raw_cache:
                return raw_cache[key]
    return None

mrna_entities["mygene_raw"] = mrna_entities.apply(map_raw, axis=1)

print("GO fetch done:", datetime.now())

In [ ]:
mrna_entities

In [ ]:
# --- Save the processed DataFrame ---
mrna_entities.to_pickle("m_rna_entities_processed.pkl")
print("Saved mrna_entities to m_rna_entities_processed.pkl")

In [ ]:
print("GO BP terms counts:", mrna_entities["GO_BP_count"].sum())
print("GO MF terms counts:", mrna_entities["GO_MF_count"].sum())
print("GO CC terms counts:", mrna_entities["GO_CC_count"].sum())

In [45]:

# Numbers of refseq and GO terms are going to be crazy but important if we check correclty to be honest.

GO BP terms counts: 1003704
GO MF terms counts: 673342
GO CC terms counts: 730160
